## 1.记忆管理策略
- 修剪消息：并不是真正的删除信息，在AgentState中的信息列表依然是完整的，只不过发给LLM之前会进行修剪，只保留一部分信息。
- 删除消息：直接删除AgentState中保存的消息，也就是历史消息不复存在了。
- 总结消息：把历史的消息利用大模型总结出摘要+最新的消息==>拼接成新的消息发给大模型。

## 2.总结消息的案例
1*.导入SummarizationMiddleware和数据库(内存)依赖  
2.初始化数据库连接  
3.初始化Checkpointer状态存储和自动建表  
4*.初始化中间件  
5.创建Agent，绑定状态存储器和中间件

In [1]:
from langchain.agents import create_agent
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain.messages import HumanMessage
from langchain.agents.middleware import SummarizationMiddleware

#连接数据库
concetion = sqlite3.connect("Database/checkpoint.db",check_same_thread=False)
#初始化Checkpointer
checkpointer = SqliteSaver(concetion)
#自动建表
checkpointer.setup()

#初始化中间件
middleware=SummarizationMiddleware(
            model="deepseek-chat",
            trigger=("messages", 3),
            keep=("messages", 1)
        )

#创建Agent
clint = create_agent(
    "deepseek-chat",
    checkpointer=checkpointer,
    middleware=[middleware]
)

#创建会话id
config = {"configurable":{"thread_id":"thread_03"}}

In [4]:
#调用3次，会有6条messages产生
clint.invoke({"messages":[HumanMessage(content="我是王者荣耀的刘禅")]},config=config)
clint.invoke({"messages":[HumanMessage(content="现在我4级了，我有大招了")]},config=config)
clint.invoke({"messages":[HumanMessage(content="我偷偷拆掉了对面的高地")]},config=config)

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user is playing Liu Shan from Honor of Kings and has reached level 4. The goal is to provide level 4+ gameplay guidance, focusing on ultimate ability usage, combos, and macro decisions.\n\n## SUMMARY\n- The user confirmed they are level 4 and have ultimate ("大招").\n- I explained that level 4 is Liu Shan\'s first power spike – ultimate unlocks enhanced tower pushing and teamfight control.\n- Key ultimate uses:\n  - Tower pushing: Ultimate + auto-attacks/skills disable towers (interference effect), making Liu Shan extremely effective at sieging.\n  - Teamfights: Ultimate provides sustained damage and slow; can weave in skill 1 and 2 during ultimate (ultimate doesn\'t interrupt other skills).\n- Level 4 tempo: Coordinate with jungler/adc for tower dives (skill 1 knockup → skill 2 stun → ultimate for tower damage + output). Priority is to push towers whenever minions are in range.\n

In [5]:
response = clint.invoke({
    "messages":[
        HumanMessage(content="你总结一下我之前跟你说的东西")
    ]
},config
)

In [6]:
print(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nUser is playing Liu Shan (Honor of Kings) and has reached level 4. The goal is to provide level 4+ gameplay guidance with focus on ultimate usage, combos, macro decisions, and converting advantages after successful tower pushes.\n\n## SUMMARY\n- User achieved a successful stealth dive and destroyed the enemy's high ground tower.\n- I congratulated and provided post-high-ground-takedown strategy:\n  - **Golden window after high ground destruction**: If minions and teammates are close, consider pushing crystal immediately (ultimate + interference + skill 2 deals massive damage). If teammates are absent, retreat immediately to avoid being caught.\n  - **Teamfight decisions after high ground destruction**: Super minions force enemies to split, creating numerical advantage. Two options: (1) Frontline siege with ultimate to destroy another high ground tower; (2) Flank from side bushes wit